# Progressive Disclosure & MCP Layered Architecture

### Source : TOOLS_CONTEXT_5/progressive_disclosure_2.ipynb


**Context Engineering with Tools & MCP on Databricks**

This notebook demonstrates the Databricks MCP layered architecture using a real external
MCP server: the **GitHub MCP server** connected through Databricks AI Gateway.

The three stages map directly to progressive disclosure:
**Discovery** : Tool names + one-line summaries only 
**Planning** : Full schemas for shortlisted tools only
**Execution** : Raw tool result (schemas evicted) 

## Setup 

- A `.env` file with `DATABRICKS_HOST` and `DATABRICKS_TOKEN`

In [ ]:
import os
from dotenv import load_dotenv

# Load credentials from .env at project root
load_dotenv()

# Verify the credentials are present
assert os.getenv("DATABRICKS_HOST"), "DATABRICKS_HOST not set — check your .env file"
assert os.getenv("DATABRICKS_TOKEN"), "DATABRICKS_TOKEN not set — check your .env file"
assert os.getenv("SQL_WAREHOUSE_ID"), "SQL_WAREHOUSE_ID not set — check your .env file"
assert os.getenv("MLFLOW_TRACKING_URI"), "MLFLOW_TRACKING_URI not set — check your .env file"


## Imports and config

In [ ]:
import json
import uuid
import mlflow
import tiktoken
from databricks.sdk import WorkspaceClient
from databricks_langchain import ChatDatabricks
import nest_asyncio
from pydantic import create_model
from databricks_mcp import DatabricksOAuthClientProvider, DatabricksMCPClient
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_core.tools import StructuredTool
from mcp import ClientSession

# Patch the running Jupyter/VS Code event loop so await works in notebook cells
nest_asyncio.apply()


In [ ]:
# update if your Unity Catalog connection has a different name
GITHUB_CONNECTION = "github_mcp"

w              = WorkspaceClient()
llm            = ChatDatabricks(endpoint="databricks-meta-llama-3-3-70b-instruct")
enc            = tiktoken.get_encoding("cl100k_base")
GITHUB_MCP_URL = f"{w.config.host}/api/2.0/mcp/external/{GITHUB_CONNECTION}"

mlflow.set_experiment("/Users/oliver@mlops-media.com/nb2_mcp_progressive_disclosure")
print(f"Workspace host  : {w.config.host}")
print(f"GitHub MCP URL  : {GITHUB_MCP_URL}")

---
# 1 - Discovery stage: what does the GitHub MCP server expose?

## 1-1 - List all available GitHub MCP tools (the full catalog)

The first thing we do is call `list_tools()` on the proxy to discover
what the GitHub MCP server exposes. This is the **raw discovery** call --
we use it here to understand the full tool surface before applying
progressive disclosure.


In [ ]:
# DatabricksMCPClient gives a synchronous interface to the proxy
github_client = DatabricksMCPClient(
    server_url=GITHUB_MCP_URL,
    workspace_client=w,
)

all_tools = github_client.list_tools()

print(f"GitHub MCP server exposes {len(all_tools)} tools")
print()
for t in all_tools:
    print(f"  {t.name:<45} {(t.description or '')[:80]}")


## 1-2 - Baseline: raw tool dump anti-pattern

If we injected ALL tool schemas into the context on every turn,
this is what it would cost.


In [ ]:
# Simulate the cost of injecting all schemas upfront (the anti-pattern)
all_schemas_text = json.dumps([
    {"name": t.name, "description": t.description, "inputSchema": t.inputSchema}
    for t in all_tools
])
tokens_all = len(enc.encode(all_schemas_text))

print("=== Baseline: raw tool dump ===")
print(f"  Tools available  : {len(all_tools)}")
print(f"  Tokens in context: ~{tokens_all:,}  (paid on EVERY turn, regardless of relevance)")
print()
print("With 40+ GitHub MCP tools, this is the constant overhead before")
print("the model has even read the user message.")


---
# 2 - Progressive disclosure in practice

## 2-1 - Stage 1 (Discovery): build the lightweight manifest

At discovery time we inject only the tool **name + a one-line summary**.
This is enough for the model to decide which tools are relevant.
The full parameter schemas are NOT loaded yet.


In [ ]:
# Build the discovery manifest: name + trimmed description only
DISCOVERY_MANIFEST = [
    {
        "name":    t.name,
        "summary": (t.description or "")[:120],   # one line max
    }
    for t in all_tools
]

manifest_text    = json.dumps(DISCOVERY_MANIFEST)
tokens_discovery = len(enc.encode(manifest_text))

print("=== Stage 1: Discovery manifest ===")
print(f"  Tokens in context: ~{tokens_discovery:,}  (~{tokens_discovery//len(all_tools)} tokens/tool)")
print(f"  vs baseline      : {100*tokens_discovery//tokens_all}% of the full schema dump")
print()
for item in DISCOVERY_MANIFEST[:8]:   # show first 8 as sample
    print(f"  {item['name']:<45} {item['summary'][:70]}")
if len(DISCOVERY_MANIFEST) > 8:
    print(f"  ... ({len(DISCOVERY_MANIFEST)-8} more tools)")


## 2-2 - Stage 1 (Discovery): the model selects which tools it needs

The model reads the lightweight manifest and returns a shortlist.
Only names and summaries are in the context window at this point.


In [ ]:
SYSTEM_MESSAGE = (
        "You are a tool router for the GitHub MCP server.\n"
        "Your ONLY job is to read the user task and return the exact tool names needed.\n"
        "\n"
        "Rules:\n"
        "  - Respond with ONLY a raw JSON array of strings. No explanation, no markdown.\n"
        "  - Use EXACT tool names from the list below (copy-paste, do not paraphrase).\n"
        "  - Select 1 to 3 tools maximum.\n"
        "  - If listing issues: use list_issues\n"
        "  - If counting commits: use list_commits\n"
        "  - If searching repositories: use search_repositories\n"
        "  - If adding a pull request review: use pull_request_review_write\n"
        "  - If reading a PR: use pull_request_read\n"
        "\n"
        "Example output for 'list open issues in a repo': [\"list_issues\"]\n"
        "Example output for 'search repos and list commits': [\"search_repositories\", \"list_commits\"]\n"
        "\n"
    )

In [ ]:
def discovery_step(task: str) -> list[str]:
    """
    Stage 1: ask the model which GitHub MCP tools it needs for the task.
    Only the lightweight manifest (names + summaries) enters the context window.
    Returns a list of tool names from DISCOVERY_MANIFEST.
    """
    # Build a readable tool index: numbered list with name and summary
    tool_index = "\n".join(
        f"  {i+1:2}. {t['name']:<45} {t['summary']}"
        for i, t in enumerate(DISCOVERY_MANIFEST)
    )

    system = SystemMessage(content=SYSTEM_MESSAGE+(f"Available tools:\n{tool_index}\n"))

    response = llm.invoke([system, HumanMessage(content=task)])

    # Strip any accidental markdown fences before parsing
    raw = response.content.strip().strip("```json").strip("```").strip()
    try:
        shortlisted = json.loads(raw)
        # Keep only names that actually exist in the manifest
        valid = {t["name"] for t in DISCOVERY_MANIFEST}
        shortlisted = [n for n in shortlisted if n in valid]
    except Exception:
        shortlisted = []
    return shortlisted


In [ ]:

test_tasks = [
    "List the open issues in the databricks/databricks-sdk-py repository",
    "Count the number of commits last month on the main branch of MLflow",
    "Add a review comment to an open pull request",
]

print("=== Stage 1: Discovery - tool shortlisting ===")
shortlists = {}
for task in test_tasks:
    shortlisted = discovery_step(task)
    shortlists[task] = shortlisted
    tokens_used = len(enc.encode(manifest_text))  # same manifest for all tasks
    print(f"  Task      : {task}")
    print(f"  Shortlist : {shortlisted}")
    print(f"  Context   : ~{tokens_used:,} tokens (manifest only, no schemas)")
    print()

## 2-3 - Stage 2 (Planning): load full schemas for shortlisted tools only

Now we fetch the complete parameter schemas, but **only for the 2-3 tools selected
in the discovery step**. Everything else stays out of the context window.


In [ ]:
async def planning_step(shortlisted: list[str]) -> tuple[list, int]:
    """
    Stage 2: fetch full tool schemas only for the shortlisted tools.
    Returns (lc_tool_objects, token_count_of_schemas).
    """
    raw_tools = github_client.list_tools()
    selected  = [t for t in raw_tools if t.name in shortlisted]
    schema_str = json.dumps([
        {"name": t.name, "description": t.description, "inputSchema": t.inputSchema}
        for t in selected
    ])
    # Wrap each MCP tool as a LangChain StructuredTool for bind_tools()
    lc_tools = []
    for mcp_t in selected:
        props    = mcp_t.inputSchema.get("properties", {})
        required = set(mcp_t.inputSchema.get("required", []))
        fields   = {
            name: (str, ... if name in required else None)
            for name in props
        }
        InputModel = create_model(f"{mcp_t.name}_input", **fields) if fields else create_model(f"{mcp_t.name}_input")
        lc_tools.append(StructuredTool(
            name=mcp_t.name,
            description=mcp_t.description or "",
            args_schema=InputModel,
            func=lambda **kw: None,   # placeholder; real call happens in execute_step
        ))
    return lc_tools, len(enc.encode(schema_str))

In [ ]:
print("=== Stage 2: Planning - full schema loading ===")
planning_results = {}
for task, shortlisted in shortlists.items():
    lc_tools, tokens_planning = await planning_step(shortlisted)
    planning_results[task] = {"tools": lc_tools, "tokens": tokens_planning}
    print(f"  Task              : {task[:65]}")
    print(f"  Shortlisted       : {shortlisted}")
    print(f"  Tokens (planning) : ~{tokens_planning:,}")
    print(f"  vs baseline       : {100*tokens_planning//max(tokens_all,1)}% of full dump")
    print()

## 2-4 - Stage 3 (Execution): call the tool, schema evicted from context

Once the model generates a tool call, only the raw result enters the context.
The schemas loaded in the planning step are no longer needed and are not forwarded.


In [ ]:
def execution_step(tool_name: str, tool_args: dict) -> tuple[str, int]:
    """
    Stage 3: call a GitHub MCP tool through the AI Gateway proxy.
    Uses DatabricksMCPClient (synchronous) -- consistent with cells 2 and 6.
    Schemas from the planning step are NOT forwarded here.
    Returns (result_text, result_token_count).
    """
    result  = github_client.call_tool(tool_name=tool_name, arguments=tool_args)
    content = result.content[0].text if result.content else f"No result from {tool_name}"
    return content, len(enc.encode(content))

In [ ]:
# Demonstrate execution for the demo task
demo_task        = test_tasks[0]
demo_shortlisted = shortlists[demo_task]
demo_tools       = planning_results[demo_task]["tools"]

if demo_shortlisted:
    model_with_tools = llm.bind_tools(demo_tools)
    response = model_with_tools.invoke([
        SystemMessage(content="You are a GitHub assistant. Call the right tool."),
        HumanMessage(content=demo_task),
    ])

    result_text, result_tokens = "", 0
    if response.tool_calls:
        tc = response.tool_calls[0]
        result_text, result_tokens = execution_step(tc["name"], tc["args"])
        print("=== Stage 3: Execution ===")
        print(f"  Tool called   : {tc['name']}")
        print(f"  Arguments     : {json.dumps(tc['args'], indent=2)}")
        print(f"  Result tokens : ~{result_tokens:,}  (only this enters context now)")
        print()
        print("  Result preview:")
        print("  " + result_text[:400].replace("\n", "\n  "))
    else:
        print("Model did not generate a tool call.")
else:
    print("No tools shortlisted for demo task.")

## 2-5 - Token comparison across all three stages

Bring all three stages together to show the progressive token savings.


In [ ]:
# For the demo task, collect all token counts
demo_tokens_discovery = tokens_discovery       # discovery manifest (shared across all tasks)
demo_tokens_planning  = planning_results[demo_task]["tokens"]
demo_result_tokens    = result_tokens if "result_tokens" in dir() else 0

print("=== Token Budget: Layered MCP vs Baseline ===")
print(f"  Task: {demo_task}")
print()
print(f"  Baseline (all schemas upfront)  : ~{tokens_all:>6,} tokens")
print(f"  Stage 1 - Discovery manifest    : ~{demo_tokens_discovery:>6,} tokens  ({100*demo_tokens_discovery//tokens_all}% of baseline)")
print(f"  Stage 2 - Planning (shortlisted): ~{demo_tokens_planning:>6,} tokens  ({100*demo_tokens_planning//max(tokens_all,1)}% of baseline)")
print(f"  Stage 3 - Execution result      : ~{demo_result_tokens:>6,} tokens  (variable)")
print()
layered_total = demo_tokens_discovery + demo_tokens_planning
saving        = 100 * (1 - layered_total / tokens_all)
print(f"  Layered total (Stage 1 + 2)     : ~{layered_total:>6,} tokens")
print(f"  Saving vs baseline              : {saving:.0f}%")
print()
print("Why it matters:")
print("  Every token in the context window competes for the model attention budget.")
print()
print("  At 50+ GitHub MCP tools, the baseline costs ~{:,} tokens before the model".format(tokens_all))
print("  has even read the user message. Progressive disclosure cuts this to the")
print("  ~{:,} tokens needed for the 2-3 actually relevant tools.".format(demo_tokens_planning))


---
# 3 - Full LangGraph agent with three explicit nodes

## 3-1 - Write the layered agent to agent_layered_mcp.py

We build a LangGraph graph with three nodes that exactly mirror the architecture:
- `discover`: fetches name+summary list, model selects relevant tools
- `plan`: fetches full schemas for selected tools only, model generates tool call
- `execute`: calls the GitHub MCP tool through the proxy, schema evicted


## 3-2 - Run the layered agent end to end

In [ ]:
import importlib, sys
import agent_layered_mcp
from agent_layered_mcp import LAYERED_AGENT

agent_tasks = [
    "Search for repos about context engineering with more than 500 stars",
]

print("=== Layered MCP Agent - GitHub MCP ===")
for task in agent_tasks:
    print(f"\nTask: {task}")
    with mlflow.start_run(run_name=f"layered_github_{uuid.uuid4().hex[:6]}"):
        mlflow.log_param("task", task)
        result = LAYERED_AGENT.invoke({
            "messages":          [{"role": "user", "content": task}],
            "shortlisted_tools": [],
            "token_log":         {},
        })
        token_log   = result.get("token_log", {})
        shortlisted = result.get("shortlisted_tools", [])
        mlflow.log_metrics({k: v for k, v in token_log.items() if isinstance(v, (int, float))})
        print(f"  Shortlisted tools : {shortlisted}")
        print(f"  Token log         : {token_log}")
        total = sum(token_log.values())
        print(f"  Total (layered)   : ~{total:,} tokens  vs  baseline ~{tokens_all:,} tokens  ({100*(1-total/max(tokens_all,1)):.0f}% saving)")
        # Print the final answer
        final = result["messages"][-1]
        if hasattr(final, "content") and final.content:
            print(f"  Answer preview    : {str(final.content)[:200]}")
